# Week 11 - CIFAR with CNNs and Quantum Tomography

- Exercise 1: We use a convolutional neural network to classify images from CIFAR and look what is going on inside the network.

- Exercise 2: We train a neural network to fit the distribution of a quantum state from few measurements.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

# Exercise 1: Convolutional Neural Networks for CIFAR10


In this exercise, you will train a simple CNN to classify images from the CIFAR10 dataset.


## Exercise 1.1
Download the CIFAR10 dataset using `torchvision.datasets.CIFAR10`, and build the train and test dataloaders, setting the batch size to 32 and activating reshuffling at each epoch for the train data by setting `shuffle=True`. Visualize some images and their different color channels.
- What does the transform do?

In [22]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

training_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


train_dataloader = ...
test_dataloader = ...

Files already downloaded and verified
Files already downloaded and verified


In [ ]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

def imshow(img):
    img = img / 2 + 0.5 # Unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()
    
## YOUR CODE HERE ##
## VISUALIZE SOME IMAGES, of different classes ##

## Exercise 1.2

 Define a function returning a convolutional neural network built with `nn.Sequential`. Use a first layer of 6 convolutional channels with filter size 5, a max-pooling layer over a $2 \times 2$ window, a second convolutional layer made of 16 channels with filter size 5, another $2 \times 2$ max-pooling layer, two dense layers with 120 and 84 neurons respectively, and a final linear layer with 10 outputs.

 Use the `nn.Flatten()` operation if needed. You can also find convolutional layers in the pytorch documentation. 
 
 Bonus: What would be other ways to implement the same convolutional model?


In [27]:
def initialize_cnn():

    ### YOUR CODE ###

    return ...

## Exercise 1.3

Using the cross-entropy loss and SGD with learning rate 0.01, train the model for 5 epochs. After training, compute the accuracy on the test set. Did your algorithm converge?

Bonus: If you are on colab, use the GPU. Compare the times -- did it help speed the process up? Don't forget to also put the batches on the GPU.

In [ ]:
model = initialize_cnn()

## YOUR TRAINING CODE HERE ##

In [ ]:
## plot the loss curve ##

## Exercise 1.3 - Changing the optimizer

Making an analogy with a physical system, we can think of the negative gradient as a force moving a particle through parameter space, following Newton’s laws. Adding a momentum or inertia term, the optimization algorithm remembers the directions of the past gradients and continues to move in their direction. Mathematically,
$$
    v_t = \gamma v_{t-1} + \eta \nabla_\theta L(\theta_t)
$$
$$
    \theta_{t+1} = \theta_t - v_t,
$$
where $\gamma \in [0,1]$ is the momentum parameter, $\eta$ the learning rate, and $\theta$ the parameters of the model. Momentum helps the optimization dynamics gain speed in directions with persistent small gradients and suppresses oscillations. Repeat training, adding `momentum=0.9` to the SGD class.

- What is the effect of the momentum (compared to no momentum)? To understand, it helps to plot the loss curves.
- Bonus: Set the momentum value very high. What happens? Does that match your expectation?

In [ ]:
model = initialize_cnn()

## YOUR TRAINING CODE HERE ##

## Exercise 1.4 - Feature Maps

We often argue that neural networks are black-boxes. We say that we do not understand what is going on inside and how they do their computations.

In fact, they are not completely black-box: We can look inside and we know *every multiplication or addition* that is happening to produce the final prediction. We just do not know a useful way to **interpret** these introspections macroscopically - there is no simple explanation that arrises directly from the matrix-multiplications.

One way to get a better insight, is to understand how the input looks after a single layer or looks from the perspective of a single neuron. E.g. for which pixel is the neuron activated high and for which low? This allows us to get an intuition which **features** activate the neuron.

**A) - Layers** Using `torch.fx`, (**f**eature e**x**traction) we can visualize these transformations of an input inside our neural network. For different input images, check the outputs of the first convolutional layer, of the first ReLU application, and of the first pooling layer.

In [40]:
from torchvision.models.feature_extraction import get_graph_node_names
from torchvision.models.feature_extraction import create_feature_extractor

nodes, _ = get_graph_node_names(model)
print(nodes) # Prints the nn.Sequential layer names

feature_extractor = create_feature_extractor(
	model, return_nodes=['0', '1', '2']) # Outputs of first conv. layer, ReLU, and first pooling layer

['input', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']


In [ ]:
dataloader = DataLoader(training_data, batch_size=32)

dataiter = iter(dataloader)
images, labels = next(dataiter)
id = 4
image = images[id]

out = feature_extractor(image.unsqueeze(0))

imshow(image)

# show the outputs after the given layers

**B) - Neurons** Can we look at what a particular neuron reacts to? What are the features learned by deep models? A simple idea to visualize these features, called activation maximization, consists in looking for the input with bounded norm that maximizes the activation of a given neuron ($x^* = \arg \max_{x: \; \|x\|=1} h_i^{\ell}(x,\theta^*)$, where $h_i^\ell$ is the activation of the neuron $i$ at layer $\ell$ of a trained network). Open https://distill.pub/2017/feature-visualization/appendix and check how the neurons in different layers of the GoogLeNet network are specializing to recognize features with various complexity, from simple textures to meaningful semantic concepts!

# Exercise 2: Neural Networks for Quantum State Tomography

In this exercise you will train a neural network to fit the distribution of a quantum state from few measurements of it. Even though this problem has applications in physics, for the purpose of this exercise we will mainly consider it as a regression problem, without concerning us with the physics behind it.

> ⚠️ You should be able to solve the exercises without GPU acceleration, but feel free to add it if you want.

## Introducing the problem: Quantum State Tomography

We are given a dataset with $N$ samples $\{x_i,y_i \}_{i = 1, ..., N}$ where $x_i = \{0,1\}^{\mathtt{n\_spins}}$ is a binary string of length $\mathtt{n\_spins}$ and $y_i \in (0,1]$ is a  number. We have our regression problem.

But where is the data from? We are considering the case where the $x_i$ are the the possible outcomes of a measurement of a quantum state $\Psi$ and $y_i$ is the probability of observing outcome $x_i$.
Our goal is to train a neural network $f_\theta(x) = \hat{y}$ that approximates the original probability distribution. Specifically, for the set of possible outcomes $\{x_i\}$, the model should return a vector of probabilities corresponding to each outcome.


## Exercise 2.1: A fully connected network

For the purpose of this exercise, we give you a function that samples $N$ measurements from a quantum state along with their probabilities.

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

def get_data(n_spins):
    
    # Loads the data available in the .txt files we made available to you.
    
    if n_spins not in [8,12,16]:
        raise ValueError(f"The data for {n_spins} is not available.")
        
    numbers = [bin(i)[2:].zfill(n_spins) for i in range(2**n_spins)]
    xs = [[int(char) for char in string] for string in numbers]
    X = np.array(xs)
    
    Wstate = np.loadtxt(f"target_state_{n_spins}.txt")
    Y = np.abs(Wstate)**2
    
    # shuffle
    idx = np.random.permutation(range(len(Y)))
    
    return X[idx], Y[idx]

a) Load the data for `n_spins=16` and visualize the labels as a histogram. Visualize both `y` and `log(y)` in different plots. Which representation is more informative?

*Answer:* ...

In [ ]:
X, Y = ...

# your code here

b) We want to continue working with this datasets in two different forms, either without or with the log-transform of the labels : $\{x_i,y_i\}$ or $\{x_i,log(y_i)\}$. Then we want to train neural networks on this data using SGD. 

Complete the function `get_loaders`. Set `n_spins=12` and apply the log transform according to the normalize variable. Create a test, train and validation dataset using `TensorDataset`, the split should be done at (40%,20%,40%) of the original data. To prepare for training, wrap the new datasets in torch `DataLoaders` as you have seen in previous exercises. For training data loader, use a batch size 32 and use shuffling, for test and validation data loaders use the full batch and don't shuffle. 

In [ ]:
def get_loaders(X, Y, normalize_log = False):
    
    n_spins = 12
   
    ... # your code here
    
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_loaders(X, Y, normalize_log=True)

c) Define a general function `train` as given below that takes the test and train loaders, a learning rate and a model, a number of epochs and optionally a boolean `logging`equal to `True` by default and returns the train and test losses for every epoch, and the best test loss achieved overall. 

You are supposed to run SGD with the MSE loss. Print the progress of your training by printing the losses every epoch, if `logging==True`.

- I) In the course of your code you will use the line `optimizer.zero_grad()`. Explain briefly what happens when you call this function, and when and why you need to call it.

- II)  Explain why we previously needed to activate the batches and shuffling for the training dataloader. 

- III) When a dataloader is used several times for several epochs, is the data shuffled in the same order every time?

*`Answers:`* ...

In [ ]:
def train(train_loader, test_loader, lr, model, n_epochs, logging=True):
    
    # your code here
    ...
    
    return train_losses, test_losses, best_test_loss

d) Test your function by training a 1 hidden layer fully connected network with ReLU activations. You will choose the hyperparameters: `n_epochs` to train and the `learning rate`. You can keep the number of `hidden_neurons=16` fixed for the moment. Hint: You should not need to use more than 2000 epochs in your explorations...

Write the code to train and visualize the test and train losses with a log-sacle on the y-axis.
It is now your turn to play.

Choose `lr=0.1`, `n_epochs=100` and `hidden_neurons=16`. Train 10 models with these hyperparameters and observe their learning curves. We would expect the learning curves of the 10 different models to be different because every time you define a new model, the weights are initialized randomly and the SGD optimizer will go over the batches in a different order. 

Describe what differences and similarities you observe. How could you change (some of) the hyperparameters so that the runs are more similar, i.e. the loss curve is less strongly dependent on the initialization?

*`Answer:`* 

In [ ]:
"""
# your code here after you ran it to answer the question, in a comment
"""

e) Choose some parameters that reliably give you good models even if you restart the training . If you get a test loss around 0.01 you can stop training. 

*`Answer:`* ..

In [ ]:
# your code here

f) For the previous model that you liked, plot the histogram of the distribution again with the log transform, for test data. Also plot the predicted and ground truth values against each other. Are you satisfied with the match?

*`Answer:`* ...

In [ ]:
# your code here

## Exercise 2.2: Exploring alternative architectures  and losses

You have found a model that works! But can you make it more efficient?

a) A linear regression model would for example use fewer parameters. Try the model and based on your experiment argue that it is less suitable for this task than the fully connected neural network.

*`Answer:`* ...

In [ ]:
# your code here

b) If a linear regression does not work, maybe you can fine-tune the number of hidden neurons. Select 5 different values of the hidden neurons `[2,8,16,32,64]` and train those models. Use at max 500 epochs to train every model and a learning rate of 0.01. In reality you would optimize the learning rate for each architecture and you can still do so if you wish, but to save you time we do not require this. After the models are trained, select the best one based on the best test loss. 

Estimate the loss you can expect on the model that you just selected on a fresh data sample, that has not been seen in the process of selecting this model. 

*`Answer:`* ...

In [ ]:
# your code here

c) Now you want to see what difference the log transformation we applied at the very beginning makes. Train a model on the plain data and compare the models prediction again both via the histogram of y-values and the scatterplot between the predicted and true y, in the log transform. What did the non-log transformed model learn and why?

*`Answer:`* ...

In [ ]:
# your code here